# Cairn: from two blueprints to twenty

The [walkthrough](walkthrough.ipynb) built everything on two projects, PFR and Carleson. This notebook follows the next step:

1. **harvesting** 20 more Lean blueprint projects;
2. making the **order optimiser** scale to large blueprints;
3. retraining the **key-declaration model** and fitting **exposition styles** on the projects that link well to Lean.

It runs in under a minute from committed results: parsed blueprints (`blueprint.json`), dependency dumps (`decls.jsonl.gz`) and reports in `results/harvest/` and `results/cross_project/`. No Lean or cloning is needed; the harvest itself is `scripts/harvest.py`. Write-ups: [`docs/harvest.md`](../docs/harvest.md) and [`docs/cross_project.md`](../docs/cross_project.md).

In [1]:
import json, gzip, random, statistics, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"
HARVEST = RES / "harvest"
pd.set_option("display.width", 160)

from cairn.blueprint import Blueprint, Node

def load_blueprint(name: str) -> Blueprint:
    """A harvested blueprint, rebuilt from its committed parse (no LaTeX sources needed)."""
    b = json.loads((HARVEST / name / "blueprint.json").read_text())
    return Blueprint([Node(**n) for n in b["nodes"]], [tuple(x) for x in b["unresolved"]],
                     [tuple(x) for x in b["orphan_proofs"]])

triage = {p.parent.name: json.loads(p.read_text()) for p in sorted(HARVEST.glob("*/triage.json"))}
print(len(triage), "projects:", ", ".join(triage))

20 projects: abc_exceptions, apap, banach_tarski, bonn_analysis, brownian_motion, cam_combi, chandra_furst_lipton, clt, con_nf, flt3, flt_regular, formal_book, infinity_cosmos, iwasawa, semicircle, sphere_eversion, sphere_packing, testing_lower_bounds, toric, zeta3


## 1. The harvest

`scripts/harvest.py` works through the projects in `scripts/harvest_projects.toml`, taken from the leanblueprint README's project list, smallest first. For each project it:

1. clones it and parses the blueprint;
2. runs the order analysis (Phase 0), which needs only the blueprint;
3. builds with the Mathlib cache and extracts dependencies;
4. compares the authors' `\uses` with Lean (Phase 1).

It keeps only the blueprint sources and the dump, and deletes the build, the cache and the toolchain before moving on, so disk use stays bounded. Each project takes 3–16 minutes.

Getting here took four fixes:
- **Old Lean.** A legacy extractor, `lean/extract_deps_legacy.lean`, handles Lean v4.7–v4.21.
- **Results moved into Mathlib.** The blueprint's `\lean` names are passed to the extractor, so upstreamed results come back as *external* declarations.
- **Stub blueprints** are skipped before building.
- **zeta3** is pinned to a commit whose live blueprint is still about ζ(3); later commits hold an unrelated IMO blueprint.

In [2]:
rows = []
for name, r in triage.items():
    rows.append({"project": name, "status": r["status"], "Lean": r.get("toolchain", "").split(":")[-1],
                 "nodes": r.get("nodes"), "uses edges": r.get("uses_edges"), "chapters": r.get("chapters"),
                 "Lean decls": r.get("decls"), "linked": r.get("nodes_linked"),
                 "linked %": round(100 * r["nodes_linked"] / r["nodes"]) if r.get("nodes") and "nodes_linked" in r else None,
                 "via Mathlib only": r.get("nodes_linked_external_only"), "extractor": r.get("extractor"),
                 "minutes": r.get("minutes")})
harvest = pd.DataFrame(rows).sort_values("nodes", ascending=False).set_index("project")
harvest

,status,Lean,nodes,uses edges,chapters,Lean decls,linked,linked %,via Mathlib only,extractor,minutes
project,,,,,,,,,,,
brownian_motion,ok,v4.33.0-rc1,663,1608,15,2222.0,492.0,74.0,183.0,extract_deps.lean,16.4
testing_lower_bounds,ok,v4.35.0-rc2,344,867,17,1035.0,177.0,51.0,59.0,extract_deps.lean,5.4
semicircle,ok,v4.24.0,205,346,5,229.0,58.0,28.0,0.0,extract_deps.lean,10.0
formal_book,ok,v4.34.0-rc2,192,103,45,507.0,69.0,36.0,0.0,extract_deps.lean,6.0
con_nf,ok,v4.21.0-rc3,159,293,8,1998.0,55.0,35.0,0.0,extract_deps_legacy.lean,5.3
toric,ok,v4.35.0-rc2,145,245,13,276.0,67.0,46.0,42.0,extract_deps.lean,3.9
sphere_packing,ok,v4.32.0,141,246,11,1466.0,91.0,65.0,18.0,extract_deps.lean,5.6
flt3,ok,v4.7.0-rc2,92,232,3,235.0,90.0,98.0,0.0,extract_deps_legacy.lean,5.4
sphere_eversion,ok,v4.34.0-rc2,73,104,5,1378.0,73.0,100.0,10.0,extract_deps.lean,5.8


Of 20 projects, 19 harvested and one stub (LeanCamCombi) was skipped. Three blueprints have no `\lean` links at all (Banach–Tarski, Infinity Cosmos, zeta3), so they support only the order analysis.

The Mathlib linking matters most for finished projects. Their results have been upstreamed, and the blueprint's `\lean` names now point into Mathlib:

In [3]:
harvest[harvest["via Mathlib only"].fillna(0) > 0][["nodes", "linked", "linked %", "via Mathlib only"]]

,nodes,linked,linked %,via Mathlib only
project,,,,
brownian_motion,663,492.0,74.0,183.0
testing_lower_bounds,344,177.0,51.0,59.0
toric,145,67.0,46.0,42.0
sphere_packing,141,91.0,65.0,18.0
sphere_eversion,73,73.0,100.0,10.0
bonn_analysis,65,13.0,20.0,7.0
clt,50,40.0,80.0,33.0
apap,49,37.0,76.0,4.0
flt_regular,45,29.0,64.0,17.0


## 2. Do authors' `\uses` match Lean?

Phase 1 projects the Lean dependency graph onto blueprint nodes and compares it with the edges the authors wrote. For each author edge:
- *direct*: it is also a direct Lean dependency;
- *implied*: it is only implied through a chain;
- *absent*: it is not in Lean at all.

*Unreported* is the share of Lean edges the author never wrote.

In [4]:
def phase1_row(name, path):
    s = json.loads(path.read_text())["stats"]
    e = s["edges"]
    n = lambda k: e[k] if isinstance(e[k], int) else len(e[k])
    a = n("author_edges")
    return {"project": name, "linked nodes": s["blueprint_nodes_with_formal_decl"], "author edges": a,
            "direct %": round(100 * n("author_direct_in_lean") / a), "absent %": round(100 * n("author_absent_from_lean") / a),
            "Lean edges": n("lean_edges"), "unreported %": round(100 * n("lean_unreported_by_author") / max(1, n("lean_edges")))}

p1 = [phase1_row("PFR", RES / "phase1/pfr/phase1.json"), phase1_row("Carleson", RES / "phase1/carleson/phase1.json")]
p1 += [phase1_row(n, HARVEST / n / "phase1.json") for n in triage
       if (HARVEST / n / "phase1.json").exists() and triage[n].get("nodes_linked", 0) >= 20]
pd.DataFrame(p1).set_index("project").sort_values("author edges", ascending=False)

,linked nodes,author edges,direct %,absent %,Lean edges,unreported %
project,,,,,,
brownian_motion,492,1058,87,11,2428,62
PFR,208,469,87,9,1172,65
testing_lower_bounds,177,316,81,16,883,71
Carleson,170,212,94,6,353,44
flt3,90,204,96,1,368,47
sphere_packing,91,123,79,20,315,69
sphere_eversion,73,104,88,12,265,65
con_nf,55,87,72,28,519,88
toric,67,86,62,33,144,63


- **Author edges are mostly real dependencies:** 62–96% are direct Lean dependencies. FLT3 matches Lean almost exactly (96%), and Brownian motion matches PFR's 87% with twice as many edges.
- **Authors cite key results, not everything:** a third to two thirds of the Lean edges go unreported. New Foundations reports least (88% unreported).
- **FormalBook is the outlier (9% direct).** 30 of its 68 blueprint-named theorems still depend on `sorry`, so their Lean proofs have no dependencies to compare:

In [5]:
rows = [json.loads(l) for l in gzip.open(HARVEST / "formal_book" / "decls.jsonl.gz", "rt")]
named = {x for n in load_blueprint("formal_book").nodes for x in n.lean_decls}
sorried = [r["name"] for r in rows if r["name"] in named and "sorryAx" in r["value_deps"]]
print(f"{len(sorried)} of {len(named)} blueprint-named FormalBook declarations use sorry, e.g. {sorried[:3]}")

30 of 68 blueprint-named FormalBook declarations use sorry, e.g. ['infinity_of_primes₆', 'book.quadratic_reciprocity.fact_B', 'book.irrational.lem_aux_i']


## 3. An optimiser that scales

The Phase 0 comparison asks how far an author's order is from the lightest valid order. It measures load as the *mean number of results held open* (stated but still needed later). The first harvest showed the old optimiser losing to the authors on Brownian motion (663 nodes). It had two problems:

- **It minimised the wrong quantity:** total edge length, not mean open. The two agree on small graphs and drift apart on large ones.
- **It started only from greedy orders**, which get poor on large graphs.

The new local search minimises mean open directly. Mean open × (n − 1) is the sum, over results, of (position of last use − position stated). Swapping two adjacent, unrelated results changes it by an amount computed from their immediate neighbours, and a test checks this against brute force. The search now also starts from the author's own order.

Here it is live on sphere packing:

In [6]:
from cairn.graph import (dependency_graph, greedy_min_open_order, local_search_order, make_dag,
                         nearest_topological_order, order_metrics)

bp = load_blueprint("sphere_packing")
g = dependency_graph(bp)
human = [n.id for n in sorted(bp.nodes, key=lambda n: n.position)]
dag, _ = make_dag(g, human)
author = nearest_topological_order(dag, human)          # the author's order, forward references repaired
rng = random.Random(0)
greedy = [greedy_min_open_order(dag, rng) for _ in range(5)]
mean_open = lambda o: order_metrics(g, o).mean_open

t = time.time()
candidates = {
    "author (as written)": human,
    "greedy (best of 5)": min(greedy, key=mean_open),
    "greedy + old search (edge length)": min((local_search_order(dag, o) for o in greedy), key=mean_open),
    "greedy + new search (mean open)": min((local_search_order(dag, o, objective="open") for o in greedy), key=mean_open),
    "author + new search (mean open)": local_search_order(dag, author, objective="open"),
}
print(f"{len(g)} nodes; searches took {time.time() - t:.1f}s")
pd.Series({k: round(mean_open(o), 1) for k, o in candidates.items()}, name="mean open")

141 nodes; searches took 0.8s


author (as written)                  15.7
greedy (best of 5)                   24.2
greedy + old search (edge length)    17.6
greedy + new search (mean open)      16.2
author + new search (mean open)      13.1
Name: mean open, dtype: float64

Phase 0 keeps two optimised orders:
- `optimised`: from greedy starts only. τ against it still says whether the author follows an *independent* optimum.
- `best_known`: also searches from the author's order. It is the reference for the **gap** below.

*Gap* = (human − best known) / (uniform random − best known). 0 means as light as the best order known, and 1 means no better than a random valid order. "Chapter gap" is the median over chapters with at least 5 nodes.

In [7]:
def gap(scope, ref="best_known"):
    h, o, u = scope["orders"]["human"]["mean_open"], scope["orders"][ref]["mean_open"], scope["uniform"]["mean_open"]["p50"]
    return (h - o) / (u - o) if u > o else float("nan")

paths = {"PFR": RES / "phase0/pfr/phase0.json", "Carleson": RES / "phase0/carleson/phase0.json"}
paths |= {n: HARVEST / n / "phase0/phase0.json" for n in triage if (HARVEST / n / "phase0/phase0.json").exists()}
rows = []
for name, path in paths.items():
    d = json.loads(path.read_text())
    w = d["scopes"][0]
    o = w["orders"]
    chapters = [gap(s) for s in d["scopes"][1:] if s["n_nodes"] >= 5]
    rows.append({"project": name, "nodes": w["n_nodes"], "human": o["human"]["mean_open"],
                 "greedy-start optimum": o["optimised"]["mean_open"], "best known": o["best_known"]["mean_open"],
                 "random": w["uniform"]["mean_open"]["p50"], "gap": gap(w),
                 "chapter gap": statistics.median(chapters) if chapters else None, "chapters": len(chapters)})
p0 = pd.DataFrame(rows).set_index("project").sort_values("nodes", ascending=False).round(2)
p0 = p0[p0["nodes"] >= 20]          # Iwasawa (15 nodes, 3 edges) is too small to score
p0

,nodes,human,greedy-start optimum,best known,random,gap,chapter gap,chapters
project,,,,,,,,
brownian_motion,663,39.67,41.99,29.69,145.06,0.09,0.21,14
testing_lower_bounds,344,32.77,25.90,25.90,61.68,0.19,0.49,12
PFR,218,31.92,27.41,26.18,55.36,0.20,0.17,8
semicircle,205,14.60,12.46,9.72,41.62,0.15,0.17,4
formal_book,192,1.20,1.46,1.02,32.84,0.01,0.52,8
Carleson,180,16.63,10.69,9.33,39.22,0.24,1.26,7
con_nf,159,15.13,13.14,13.14,29.41,0.12,0.21,7
toric,145,17.73,9.93,9.93,36.22,0.30,0.80,7
sphere_packing,141,15.67,13.08,13.08,33.22,0.13,0.11,5


In [8]:
ratio = p0["human"] / p0["best known"]
print(f"median gap {p0['gap'].median():.2f}; {int((p0['gap'] <= 0.3).sum())} of {len(p0)} blueprints within 0.3")
print(f"authors hold {ratio.median() - 1:.0%} more results open than the best known (median)")
print("search from the author's order beat every greedy start on:",
      ", ".join(p0.index[p0["best known"] < p0["greedy-start optimum"] - 1e-9]))

median gap 0.18; 16 of 20 blueprints within 0.3
authors hold 25% more results open than the best known (median)
search from the author's order beat every greedy start on: brownian_motion, PFR, semicircle, formal_book, Carleson, apap, flt_regular, banach_tarski


- **Authors order whole documents much closer to the optimum than to random**, but not at it. The median author order holds about a quarter more results open than the best order known. The old optimiser put PFR within 5% of optimal; that was too generous.
- **Within chapters the two styles from the walkthrough reappear.** Most projects have chapter gaps around 0.1–0.3 (dependency order). Carleson, ABC, Bonn and Chandra–Furst–Lipton are about 1, like random, because they follow the argument of a paper.
- **Starting from the author's order often finds the best order known.** An author's draft is a good starting point for making the order easier on readers.

## 4. Key-declaration model on 9 projects

The model that picks which declarations to name was trained on PFR and Carleson only. It is now trained on the 9 projects whose blueprints are mostly linked (at least half the nodes) and name at least 30 declarations inside the project. Each is scored by a model trained on the other eight (*leave one project out*). The other harvested projects are scored as held out. The *baseline* is the old PFR + Carleson model.

In [9]:
loo = json.loads((RES / "cross_project" / "key_node_loo.json").read_text())
key = pd.DataFrame([{"project": n, "role": "held out" if r["heldout"] else "train", "named": r["positives"],
                     "decls": r["decls"], "AUROC": r["auroc"], "P@k": r["p_at_k"],
                     "baseline AUROC": r["baseline"]["auroc"], "baseline P@k": r["baseline"]["p_at_k"]}
                    for n, r in loo.items()]).set_index("project").round(2)
display(key)
key.groupby("role")[["AUROC", "baseline AUROC", "P@k", "baseline P@k"]].mean().round(3)

,role,named,decls,AUROC,P@k,baseline AUROC,baseline P@k
project,,,,,,,
PFR,train,248,1395,0.82,0.57,0.79,0.50
Carleson,train,236,3422,0.86,0.34,0.85,0.30
brownian_motion,train,325,2222,0.70,0.30,0.68,0.28
testing_lower_bounds,train,126,1035,0.67,0.26,0.63,0.24
sphere_packing,train,100,1468,0.79,0.33,0.74,0.30
flt3,train,90,235,0.73,0.57,0.68,0.54
sphere_eversion,train,80,1378,0.86,0.32,0.82,0.30
abc_exceptions,train,49,248,0.81,0.49,0.77,0.43
apap,train,34,769,0.93,0.65,0.89,0.68


,AUROC,baseline AUROC,P@k,baseline P@k
role,,,,
held out,0.746,0.666,0.272,0.246
train,0.797,0.761,0.426,0.397


Mean AUROC rises from 0.76 to 0.80 on the training projects and from 0.67 to 0.75 on the held-out ones. Three held-out projects get slightly worse: semicircle, FormalBook and CLT. They are the ones with unreliable labels (partial linking, `sorry`, very few named results).

The model trained on all 9, `results/outline/key_model_all.json`, is the one to use on a new project. Here it scores a project it never saw, Toric, from the committed dump:

In [10]:
import tempfile
from sklearn.metrics import roc_auc_score
from cairn.formal import join_blueprint, load_decls
from cairn.outline import KeyModel

with tempfile.NamedTemporaryFile("wb", suffix=".jsonl", delete=False) as f:
    f.write(gzip.open(HARVEST / "toric" / "decls.jsonl.gz").read())
decls = load_decls(f.name)                       # project declarations only (external records are skipped)
scores = KeyModel.load(RES / "outline" / "key_model_all.json").score(decls)
named = {d for ds in join_blueprint(load_blueprint("toric"), decls).node_decls.values() for d in ds}
names = sorted(scores)
print(f"Toric: {len(named)} named of {len(names)} declarations; "
      f"AUROC {roc_auc_score([v in named for v in names], [scores[v] for v in names]):.2f}")
print("top 5:", [(v, v in named) for v in sorted(names, key=lambda v: -scores[v])[:5]])

Toric: 29 named of 276 declarations; AUROC 0.71
top 5: [('SO2Ring.algHomMulEquiv', False), ('AlgebraicGeometry.Scheme.isPerfPair_charPairing', True), ('AlgebraicGeometry.ToricVariety', True), ('SO2Ring.baseChangeBialgEquiv', False), ('AlgebraicGeometry.Scheme.diagHomEquiv', True)]


## 5. Exposition styles

The style fit ([`docs/style.md`](../docs/style.md)) finds the style parameters that best reproduce each author's order:
- `top_down`: how many results are stated as goals before their proofs;
- a chapter roadmap;
- definitions up front or just in time.

*Motivated* is the share of lemmas stated after a goal they serve.

In [11]:
style_paths = {"PFR": RES / "style/pfr/style.json", "Carleson": RES / "style/carleson/style.json"}
style_paths |= {n: HARVEST / n / "style/style.json" for n in triage if (HARVEST / n / "style/style.json").exists()}
rows = []
for name, path in style_paths.items():
    s = json.loads(path.read_text())
    rows.append({"project": name, "motivated": round(s["human"]["motivated"], 2),
                 "best τ (chapters)": f"{s['fit']['best_tau']['label']} ({s['fit']['best_tau']['tau']:+.2f})",
                 "closest profile (document)": s["document"]["fit"]["closest_profile"]["label"]})
pd.DataFrame(rows).set_index("project").sort_values("motivated")

,motivated,best τ (chapters),closest profile (document)
project,,,
apap,0.00,top_down=0 (+0.57),top_down=0
flt3,0.01,top_down=0 (+0.42),top_down=0
PFR,0.02,top_down=0 (+0.55),top_down=0
sphere_packing,0.07,"top_down=0, definitions=upfront (+0.55)",top_down=0
brownian_motion,0.09,"top_down=0.2, definitions=upfront, goals by ke...",top_down=0
abc_exceptions,0.19,"top_down=0.6, definitions=upfront (+0.67)","top_down=0.2, chapter roadmap"
testing_lower_bounds,0.36,"top_down=0, definitions=upfront (+0.31)",top_down=0
sphere_eversion,0.39,"top_down=0.2, definitions=upfront (+0.44)",top_down=0.2
Carleson,0.67,"top_down=0.6, chapter roadmap (+0.20)","top_down=0.2, chapter roadmap"


- **Bottom-up is the norm.** By closest profile, five of the seven new projects match PFR's pure bottom-up style.
- **Carleson is the most goal-first author.** Sphere eversion, testing lower bounds and ABC sit in between.
- **Definitions up front is common** among the new projects. PFR's "introduce definitions just before use" was PFR's habit, not a general one.

## Where this leaves us

- **19 blueprint projects** harvested, 9 of them well enough linked to train on.
- **An optimiser that scales to 600+ nodes.** It shows that authors are near, but not at, the load optimum.
- **A key-declaration model trained on 9 projects**, better on every one of them when each is held out in turn.

Next:
1. Outline the new projects with the 9-project model and score the outlines against their blueprints.
2. Use the best-known order as a reading aid: show which moves from the author's order save the most working memory.
3. Harvest the three large projects (PNT+, FLT, Equational Theories).